In [44]:
# -*- coding: utf-8 -*-
"""
@author: Jesper
"""
import re
import pandas as pd
import numpy as np
import xy.pyplot as plt
import scipy
# import os
# from matplotlib.animation import FuncAnimation
# import time
from scipy.optimize import curve_fit
from pathlib import Path
# from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from IPython.display import clear_output
import gc


from matplotlib.ticker import FuncFormatter

def engineering_formatter(x, pos):
    if x >= 1e6:
        return f"{x/1e6:g}M"
    elif x >= 1e3:
        return f"{x/1e3:g}k"
    else:
        return f"{x:g}"

# ------------------ Plot settings ------------------
plt.rcParams['figure.figsize'] = (10, 6)        # Default figure size
plt.rcParams['figure.dpi'] = 100                # Resolution
plt.rcParams['font.size'] = 16                  # Font size
plt.rcParams['axes.titlesize'] = 14             # Title font size
plt.rcParams['axes.labelsize'] = 22             # Axis label font size
plt.rcParams['xtick.labelsize'] = 19            # X-tick label size
plt.rcParams['ytick.labelsize'] = 19            # Y-tick label size
plt.rcParams['legend.fontsize'] = 22            # Legend font size
plt.rcParams['lines.linewidth'] = 2             # Line width
plt.rcParams['axes.grid'] = True                # Show grid by default
plt.rcParams['grid.alpha'] = 0.3                # Grid transparency

# Set global tick mark (line) sizes
plt.rcParams['xtick.major.size'] = 8      # Major tick length (horizontal)
plt.rcParams['ytick.major.size'] = 8      # Major tick length (vertical)
plt.rcParams['xtick.minor.size'] = 4      # Minor tick length (horizontal)
plt.rcParams['ytick.minor.size'] = 4      # Minor tick length (vertical)

# Set tick width
plt.rcParams['xtick.major.width'] = 1.5   # Major tick width (horizontal)
plt.rcParams['ytick.major.width'] = 1.5   # Major tick width (vertical)
plt.rcParams['xtick.minor.width'] = 1.5   # Major tick width (horizontal)
plt.rcParams['ytick.minor.width'] = 1.5   # Major tick width (vertical)
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["Palatino", "Book Antiqua", "DejaVu Serif"],
})

In [45]:
def correct_iq(I, Q, return_params=False):

    I = np.asarray(I, dtype=float)
    Q = np.asarray(Q, dtype=float)
 
    dc_I = np.mean(I)
    dc_Q = np.mean(Q)
    I1 = I - dc_I
    Q1 = Q - dc_Q
 
    gain_I = np.sqrt(np.mean(I1 ** 2))
    gain_Q = np.sqrt(np.mean(Q1 ** 2))
 
    I2 = I1 / gain_I
    Q2 = Q1 / gain_Q
    
    cross = np.mean(I2 * Q2)
    sin_delta = np.clip(cross, -1.0, 1.0)
    delta = np.arcsin(sin_delta)
 
    cos_delta = np.cos(delta)

    I_corr = I2
    Q_corr = (Q2 - I2 * np.sin(delta)) / cos_delta
 
    if return_params:
        params = {
            'dc_I':      dc_I,
            'dc_Q':      dc_Q,
            'gain_I':    gain_I,
            'gain_Q':    gain_Q,
            'delta':     delta,
            'delta_deg': np.degrees(delta),
        }
        return I_corr, Q_corr, params
 
    return I_corr, Q_corr

In [46]:
def load_commercial_fn(filepath):

    # Read header
    with open(filepath, encoding="ISO-8859-1") as f:

        header = []

        while True:

            line = f.readline()

            if not line:
                break

            header.append(line)

            if "Frequency" in line:
                break

    # Detect units
    unit = None

    for line in header:

        if "Hz² / Hz" in line:
            unit = "Hz2_per_Hz"

        elif "Hz/sqrt(Hz)" in line:
            unit = "Hz_per_sqrtHz"

    # Read data
    df = pd.read_csv(
        filepath,
        encoding="ISO-8859-1",
        comment="#"
    )

    freq = df.iloc[:,0].to_numpy()
    fn = df.iloc[:,1].to_numpy()

    # Remove duplicated frequencies
    idx = np.unique(
        freq,
        return_index=True
    )[1]

    freq = freq[idx]
    fn = fn[idx]

    # Convert if needed
    if unit == "Hz_per_sqrtHz":
        fn = fn**2

    return freq, fn

In [47]:
def read_metadata(filename):
    meta = {}

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()

            if ":" not in line:
                continue

            parts = line.split(":")
            
            # Most entries are Key:Value:
            if len(parts) >= 2:
                key = parts[0]
                value = parts[1]

                meta[key] = value

    # --------------------------------------------------
    # Extract powers from filename
    # Example:
    # PPCL550_Dither_14.66dBm_7.51+12.14dBm_no1.csv
    # --------------------------------------------------

    basename = filename.split("/")[-1]
    basename = basename.split("\\")[-1]

    match = re.search(
        r'([-+]?\d+(?:\.\d+)?)dBm_([-+]?\d+(?:\.\d+)?)\+([-+]?\d+(?:\.\d+)?)dBm',
        basename
    )


    if match:
        meta["TotalOutputPower_dBm"] = float(match.group(1))
        meta["SignalPower_dBm"] = float(match.group(2))
        meta["LOPower_dBm"] = float(match.group(3))

    meta["MeasurementName"] = basename


    return meta

def sliding_window_variance_analysis(
        time,
        phase_inc,
        Twin_list=None,
        step_fraction=0.1):

    if Twin_list is None:

        Twin_list = np.logspace(
            np.log10(0.05e-3),
            np.log10(30e-3),
            20
        )

    mean_var = np.full(len(Twin_list), np.nan)
    std_var = np.full(len(Twin_list), np.nan)
    n_win = np.zeros(len(Twin_list), dtype=int)

    all_var = []
    all_start_times = []

    for k, Twin in enumerate(Twin_list):

        step = step_fraction * Twin

        t_start = np.arange(
            time[0],
            time[-1] - Twin,
            step
        )

        var_local = np.full(len(t_start), np.nan)

        for m, t0 in enumerate(t_start):

            mask = (
                (time >= t0)
                &
                (time < t0 + Twin)
            )

            yy = phase_inc[mask]

            if len(yy) > 10:

                # MATLAB: var(yy,1)
                var_local[m] = np.var(
                    yy,
                    ddof=0
                )

        valid = np.isfinite(var_local)

        all_var.append(var_local[valid])
        all_start_times.append(t_start[valid])

        mean_var[k] = np.mean(var_local[valid])
        std_var[k] = np.std(var_local[valid])

        n_win[k] = np.sum(valid)


    return {
        "Twin_list": Twin_list,
        "mean_var": mean_var,
        "std_var": std_var,
        "n_windows": n_win,
        "all_var": all_var,
        "all_start_times": all_start_times
    }

In [48]:
def read_and_process_csv(filename = None, foldername = None, path = None, offset_handling = False):
    """ Reads and processes a CSV file containing time and channel data. It handles metadata, scaling, and offset adjustments based on the provided parameters.
        Input is a filename without suffix, a foldername, and an optional path. If the path is provided, it will be used instead of constructing the file path from the foldername and filename. The function reads the metadata from a corresponding .csv file and applies scaling and offset adjustments to the channel data if necessary. It returns the time array, channel data arrays, and the size of the data.
    """

    if path == None:
        file_path = fr'{foldername}/{filename}.csv'
    else:
        file_path = path
    print(file_path)

    meta = read_metadata(file_path[:-8] + ".csv") #since data file is filename + .wfm.csv and metadata file is + .csv

    dt = float(meta["SignalResolution"])
    signal_format = meta.get("SignalFormat", "").upper()
    is_raw_adc =  signal_format.startswith("INT") 

    if is_raw_adc:
        no_of_quant = int(meta["NofQuantisationLevels"])
        y_volt_start = float(meta["BaseYStart"])
        y_volt_stop = float(meta["BaseYStop"])

        scale = (y_volt_stop - y_volt_start) / (no_of_quant)

        if offset_handling:        
            vertical_offset = float(meta["VerticalOffset"]) 
        else:
            vertical_offset = 0.0

        
    print("now proccessing file at path: ", file_path)
    df = pd.read_csv(file_path, header=None, delimiter=';')
    if len(df.columns) == 5:
        df.columns = ['time', 'ch_1', 'ch_2', 'ch_3', 'ch_4']
        time = df['time']

        if is_raw_adc:
            ch_one = df['ch_1'] * scale + vertical_offset
            ch_two = df['ch_2'] * scale + vertical_offset
            ch_three = df['ch_3'] * scale + vertical_offset
            ch_four = df['ch_4'] * scale + vertical_offset
        else:
            ch_one = df['ch_1']    
            ch_two = df['ch_2']    
            ch_three = df['ch_3']    
            ch_four = df['ch_4']

        size = len(ch_one)
        print("Size: ", size)
        print("Frequency resolution: ", 1/(time[1] - time[0]))
        
        return time[0:size], ch_one[0:size], ch_two[0:size], ch_three[0:size], ch_four[0:size], size

    elif len(df.columns) == 4:
            df.columns = ['ch_1', 'ch_2', 'ch_3', 'ch_4']

            if is_raw_adc:
                ch_one = df['ch_1'] * scale + vertical_offset
                ch_two = df['ch_2'] * scale + vertical_offset
                ch_three = df['ch_3'] * scale + vertical_offset
                ch_four = df['ch_4'] * scale + vertical_offset
            else:
                ch_one = df['ch_1']    
                ch_two = df['ch_2']    
                ch_three = df['ch_3']    
                ch_four = df['ch_4']
            size = len(ch_one)

            time = np.arange(size) * dt

            # df['time'] = time #Adding time column to the dataframe
      

            print("Size: ", size)
            print("Frequency resolution: ", 1/(time[1] - time[0]))
            
            return time[0:size], ch_one[0:size], ch_two[0:size], ch_three[0:size], ch_four[0:size], size
    
        
    
def pnpsd(filenames, foldername, averaging_number):
    """
    Calculates sdd for phase noise power spectral density using cross spectral density csd. Returns the PN-PSD. 
    """   
    timeX, ch_one, ch_two, ch_three, ch_four, size = read_and_process_csv(filenames, foldername)

    NN=size//averaging_number               # // to make sure that it gets an integer and not a float
    window = scipy.signal.get_window('hann', NN)

    psdxx0=0
    fxx=0

    psdyy0=0
    fyy=0

    psdxy0=0
    fxy=0

    ch_one, ch_two = correct_iq(ch_one, ch_two)
    ch_three, ch_four = correct_iq(ch_three, ch_four)

    phaseX,sampling_rateX=pp(ch_one, ch_two, timeX)
    phaseY,sampling_rateY=pp(ch_three, ch_four, timeX)

    ffxx = 0
    Pxx_den = 0

    ffxy = 0
    Pxy_den = 0

    ffyy = 0
    Pyy_den = 0

    ffxx, Pxx_den = scipy.signal.csd(phaseX, phaseX, fs=sampling_rateX,window=window,nperseg=NN)
    psdxx0=psdxx0 + Pxx_den
    fxx=ffxx+fxx

    ffxy, Pxy_den = scipy.signal.csd(phaseX, phaseY, fs=sampling_rateX,window=window,nperseg=NN)
    psdxy0=psdxy0 + Pxy_den
    fxy=ffxy+fxy

    ffyy, Pyy_den = scipy.signal.csd(phaseY, phaseY, fs=sampling_rateX,window=window,nperseg=NN)
    psdyy0=psdyy0 + Pyy_den
    fyy=ffyy+fyy

    psdxx0=np.abs(psdxx0)

    psdyy0=np.abs(psdyy0)

    psdxy0=np.abs(psdxy0)

    # Shitty namings, but it returns frequency and PN-PSD for the x-polarization, y-polarization and the cross correlated

    return fxx, psdxx0, fyy, psdyy0, fxy, psdxy0



    
def pnpsd_alt(phaseX, phaseY, sampling_rate, size, averaging_number):
    """
    Calculates sdd for phase noise power spectral density using cross spectral density csd. Returns the PN-PSD. 
    """   

    NN=size//averaging_number               # // to make sure that it gets an integer and not a float
    window = scipy.signal.get_window('hann', NN)

    psdxx0=0
    fxx=0

    psdyy0=0
    fyy=0

    psdxy0=0
    fxy=0


    ffxx = 0
    Pxx_den = 0

    ffxy = 0
    Pxy_den = 0

    ffyy = 0
    Pyy_den = 0

    ffxx, Pxx_den = scipy.signal.csd(phaseX, phaseX, fs=sampling_rate,window=window,nperseg=NN)
    psdxx0=psdxx0 + Pxx_den
    fxx=ffxx+fxx

    ffxy, Pxy_den = scipy.signal.csd(phaseX, phaseY, fs=sampling_rate,window=window,nperseg=NN)
    psdxy0=psdxy0 + Pxy_den
    fxy=ffxy+fxy

    ffyy, Pyy_den = scipy.signal.csd(phaseY, phaseY, fs=sampling_rate,window=window,nperseg=NN)
    psdyy0=psdyy0 + Pyy_den
    fyy=ffyy+fyy

    psdxx0=np.abs(psdxx0)

    psdyy0=np.abs(psdyy0)

    psdxy0=np.abs(psdxy0)

    # Shitty namings, but it returns frequency and PN-PSD for the x-polarization, y-polarization and the cross correlated

    return fxx, psdxx0, fyy, psdyy0, fxy, psdxy0



def pnpsd_noQAM(filenames, foldername, averaging_number):
    """
    Calculates sdd for phase noise power spectral density using cross spectral density csd. Returns the PN-PSD. 
    """   
    timeX, ch_one, ch_two, ch_three, ch_four, size = read_and_process_csv(filenames, foldername)

    NN=size//averaging_number               # // to make sure that it gets an integer and not a float
    window = scipy.signal.get_window('hann', NN)

    psdxx0=0
    fxx=0

    psdyy0=0
    fyy=0

    psdxy0=0
    fxy=0

    phaseX,sampling_rateX=pp(ch_one, ch_two, timeX)
    phaseY,sampling_rateY=pp(ch_three, ch_four, timeX)

    ffxx = 0
    Pxx_den = 0

    ffxy = 0
    Pxy_den = 0

    ffyy = 0
    Pyy_den = 0

    ffxx, Pxx_den = scipy.signal.csd(phaseX, phaseX, fs=sampling_rateX,window=window,nperseg=NN)
    psdxx0=psdxx0 + Pxx_den
    fxx=ffxx+fxx

    ffxy, Pxy_den = scipy.signal.csd(phaseX, phaseY, fs=sampling_rateX,window=window,nperseg=NN)
    psdxy0=psdxy0 + Pxy_den
    fxy=ffxy+fxy

    ffyy, Pyy_den = scipy.signal.csd(phaseY, phaseY, fs=sampling_rateX,window=window,nperseg=NN)
    psdyy0=psdyy0 + Pyy_den
    fyy=ffyy+fyy

    psdxx0=np.abs(psdxx0)

    psdyy0=np.abs(psdyy0)

    psdxy0=np.abs(psdxy0)

    return fxx, psdxx0, fyy, psdyy0, fxy, psdxy0

def pp(XI,XQ,time):
    """
    Returns the accumulated phase using the arctan2 function and the sampling rate by
    """
    phase= np.unwrap(np.arctan2(XQ, XI))
    dt=time[2]-time[1]
    sampling_rate=1/dt
    return phase, sampling_rate

def fnpsd(fxy, psdxy, FSR):
    # fsdd for frequency noise power spectral density. Returns the FN-PSD.

    fsdxy = []

    for i, _ in enumerate(fxy):
        fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])
            
    return np.array(fxy),np.array(fsdxy)


# def find_fsr(freq, pn_psd, delay_est, upper_lim, peak_skips=0, show_detection = False):

#     # Function for finding the FSR from the dips, that is needed for the PN --> FN conversion. Returns the average distance 
#     # between the dips as the FSR and the peaks that are detected.

#     # Since 

#     def model_plus(f, A, B, C):
#         return A / (f**2) + B / (f**3) + C

#     def fit_one_over_f_squared_plus(f, y):
#         popt, pcov = curve_fit(model_plus, f, y, p0=(6e2, 1e9, 1))
#         return popt, pcov

#     f_min = (0.75)/(delay_est / (3e8/1.46))
#     mask = (freq < upper_lim) & (freq > f_min)
#     f_sub = freq[mask]
#     psd_sub = -pn_psd[mask]

#     delta_f = f_min
#     df = freq[1] - freq[0]

#     delta_i = int(round(delta_f / df))

#     peaks, props = scipy.signal.find_peaks(psd_sub, distance = delta_i*0.9)

#     peak_temp = f_sub[peaks][peak_skips:]
#     psd_temp = -psd_sub[peaks][peak_skips:]

#     f_dist_peaks = np.median(np.diff(peak_temp))

#     popt, pcov = fit_one_over_f_squared_plus(f_sub[peaks][peak_skips:], -psd_sub[peaks][peak_skips:])

#     if show_detection:
#         fig, ax = plt.subplots(figsize =(20, 7.5))
#         ax.loglog(freq, pn_psd, label = "Input signal")
#         #ax.loglog(freq, model(freq, popt[0], popt[1]), label = "1/f^2 fit")

#         ax.scatter(peak_temp, psd_temp, label = "Detected fringes", color = "red")

#         ax.loglog(freq, model_plus(freq, popt[0], popt[1], popt[2]), linestyle = "dashed", color = "black")

#         ax.vlines(f_min, min(pn_psd), max(pn_psd), label = "Limits for detection", linestyles="dashed", color = "black")

#         ax.vlines(upper_lim, min(pn_psd), max(pn_psd), linestyles="dashed", color = "black")

#         print("Params from fit:")
#         print("A = ", popt[0])
#         print("B = ", popt[1])
#         print("C = ", popt[2])

#         ax.set_title("Detected dips in the PN-FSD for FSR determination", fontsize = 25)
#         ax.set_ylabel(rf"PN-FSD [rad$^2$/Hz]")
#         ax.set_xlabel("Fourier frequencies [Hz]")
#         ax.set_xlim(1e2, 1e7)
#         ax.legend(loc="upper right")
        
#     return f_dist_peaks, f_sub[peaks], -psd_sub[peaks]


def find_fsr(
        freq,
        pn_psd,
        delay_est,
        upper_lim,
        peak_skips=0,
        show_detection=False):

    """
    Determine FSR from the dips in the PN PSD.

    For long delays (e.g. 700 m), multiple dips are available and
    the FSR is estimated as the median spacing between dips.

    For short delays (e.g. 30 m), only one dip may be present in the
    frequency range. In that case the dip frequency itself is used
    as the FSR estimate.
    """

    def model_plus(f, A, B, C):
        return A / (f**2) + B / (f**3) + C

    def fit_one_over_f_squared_plus(f, y):
        popt, pcov = curve_fit(
            model_plus,
            f,
            y,
            p0=(6e2, 1e9, 1)
        )
        return popt, pcov

    # ----------------------------------
    # Search region
    # ----------------------------------

    f_min = 0.75 / (delay_est / (3e8 / 1.46))

    mask = (
        (freq < upper_lim)
        &
        (freq > f_min)
    )

    f_sub = freq[mask]
    psd_sub = -pn_psd[mask]

    # ----------------------------------
    # Expected spacing between dips
    # ----------------------------------

    delta_f = f_min
    df = freq[1] - freq[0]

    delta_i = int(round(delta_f / df))

    peaks, props = scipy.signal.find_peaks(
        psd_sub,
        distance=0.9 * delta_i
    )

    peak_temp = f_sub[peaks][peak_skips:]
    psd_temp = -psd_sub[peaks][peak_skips:]

    n_peaks = len(peak_temp)

    print(f"Detected {n_peaks} fringe dips.")

    # ----------------------------------
    # Determine FSR
    # ----------------------------------

    if n_peaks >= 2:

        f_dist_peaks = np.median(
            np.diff(peak_temp)
        )

    elif n_peaks == 1:

        f_dist_peaks = peak_temp[0]

        print(
            "Only one dip detected. "
            "Using dip frequency as FSR."
        )

    else:

        raise ValueError(
            "No FSR dips detected."
        )

    # ----------------------------------
    # Optional fit
    # ----------------------------------

    popt = None
    pcov = None

    if n_peaks >= 4:

        try:

            popt, pcov = fit_one_over_f_squared_plus(
                peak_temp,
                psd_temp
            )

        except Exception as err:

            print(
                "Fit failed:"
            )

            print(err)

            popt = None
            pcov = None

    # ----------------------------------
    # Plot
    # ----------------------------------

    if show_detection:

        fig, ax = plt.subplots(
            figsize=(20, 7.5)
        )

        ax.loglog(
            freq,
            pn_psd,
            label="Input signal"
        )

        ax.scatter(
            peak_temp,
            psd_temp,
            color="red",
            label="Detected fringes"
        )

        if popt is not None:

            ax.loglog(
                freq,
                model_plus(
                    freq,
                    popt[0],
                    popt[1],
                    popt[2]
                ),
                linestyle="dashed",
                color="black",
                label="Fit"
            )

            print("Params from fit:")
            print("A =", popt[0])
            print("B =", popt[1])
            print("C =", popt[2])

        ax.axvline(
            f_min,
            color="black",
            linestyle="dashed",
            label="Detection limits"
        )

        ax.axvline(
            upper_lim,
            color="black",
            linestyle="dashed"
        )

        ax.set_title(
            f"Detected dips, FSR = {f_dist_peaks/1e6:.3f} MHz",
            fontsize=25
        )

        ax.set_ylabel(
            r"PN PSD [rad$^2$/Hz]"
        )

        ax.set_xlabel(
            "Fourier frequency [Hz]"
        )

        ax.set_xlim(
            max(freq[1], 1e2),
            max(freq)
        )

        ax.legend(
            loc="upper right"
        )

    return (
        f_dist_peaks,
        peak_temp,
        psd_temp
    )

In [49]:
COMMERCIAL_CACHE = {}

def analyze_data(
        filename,
        foldername,
        averaging_number,
        delay = 700,
        offset_handling=False,
        Nick_analysis = False):


    file_path = fr"{foldername}/{filename}.csv"

    meta = read_metadata(file_path[:-8] + ".csv")
    dt = float(meta["SignalResolution"])

    # --------------------------------------------------
    # Read waveform
    # --------------------------------------------------
   

    time, Ix, Qx, Iy, Qy, size = read_and_process_csv(
        filename=filename,
        foldername=foldername,
        offset_handling=offset_handling
    )

    # Keep original traces
    Ix_raw = Ix.copy()
    Qx_raw = Qx.copy()
    Iy_raw = Iy.copy()
    Qy_raw = Qy.copy()

    # --------------------------------------------------
    # IQ correction
    # --------------------------------------------------

    Ix_corr, Qx_corr = correct_iq(Ix, Qx)
    Iy_corr, Qy_corr = correct_iq(Iy, Qy)

    # --------------------------------------------------
    # Incremental phase, sampling rate
    # --------------------------------------------------

    phase_inc_x, fs = pp(Ix_corr, Qx_corr, time)
    phase_inc_y, _  = pp(Iy_corr, Qy_corr, time)

    # --------------------------------------------------
    # Sliding window, variance analysis
    # --------------------------------------------------

    if Nick_analysis:
        variance_x = sliding_window_variance_analysis(
            time,
            phase_inc_x
        )

        variance_y = sliding_window_variance_analysis(
            time,
            phase_inc_y
        )
        
    # --------------------------------------------------
    # Autocorrelation
    # --------------------------------------------------

    autocorr_x = scipy.signal.correlate(
        phase_inc_x,
        phase_inc_x,
        mode="same"
    )

    autocorr_y = scipy.signal.correlate(
        phase_inc_y,
        phase_inc_y,
        mode="same"
    )

    lags = scipy.signal.correlation_lags(
        len(phase_inc_x),
        len(phase_inc_x),
        mode="same"
    )
    lags_time = lags.copy() * dt

        # --------------------------------------------------
    # Autocorrelation centered
    # --------------------------------------------------

    phase_centered_x = phase_inc_x - np.mean(phase_inc_x)
    phase_centered_y = phase_inc_y - np.mean(phase_inc_y)

    autocorr_x_centered = scipy.signal.correlate(
        phase_centered_x,
        phase_centered_x,
        mode="same"
    )

    autocorr_y_centered = scipy.signal.correlate(
        phase_centered_y,
        phase_centered_y,
        mode="same"
    )


    # --------------------------------------------------
    # PN PSD
    # --------------------------------------------------

    fxx, pn_x, fyy, pn_y, fxy, pn_xy = pnpsd_alt(phase_inc_x, phase_inc_y, fs, size, averaging_number)

    # --------------------------------------------------
    # FSR
    #---------------------------------------------------

    
    

    FSR, peaks_freq, peaks_psd = find_fsr(fxx, pn_x, delay_est=delay, upper_lim=9e6)

    # --------------------------------------------------
    # FN PSD
    # --------------------------------------------------

    _, fn_x = fnpsd(fxx, pn_x, FSR)
    _, fn_y = fnpsd(fyy, pn_y, FSR)
    _, fn_xy = fnpsd(fxy, pn_xy, FSR)


    # --------------------------------------------------
    # Time-domain dataframe
    # --------------------------------------------------

    df_time = pd.DataFrame({
        "time": time,

        "Ix_raw": Ix_raw,
        "Qx_raw": Qx_raw,
        "Iy_raw": Iy_raw,
        "Qy_raw": Qy_raw,

        "Ix_corr": Ix_corr,
        "Qx_corr": Qx_corr,
        "Iy_corr": Iy_corr,
        "Qy_corr": Qy_corr,

        "phase_inc_x": phase_inc_x,
        "phase_inc_y": phase_inc_y,
    })


    if Nick_analysis:
        # --------------------------------------------------
        # Variance dataframe
        # -----------

        df_variance = pd.DataFrame({
        "Twin": variance_x["Twin_list"],
        "Twin_ms": variance_x["Twin_list"] * 1e3,

        "mean_var_x": variance_x["mean_var"],
        "std_var_x": variance_x["std_var"],

        "mean_var_y": variance_y["mean_var"],
        "std_var_y": variance_y["std_var"],

        "n_windows_x": variance_x["n_windows"],
        "n_windows_y": variance_y["n_windows"]
        })


        # --------------------------------------------------
        # Variance details dataframe
        # -----------

        variance_details = {
            "Twin_list": variance_x["Twin_list"],

            "start_times_x": variance_x["all_start_times"],
            "start_times_y": variance_y["all_start_times"],

            "local_variances_x": variance_x["all_var"],
            "local_variances_y": variance_y["all_var"]
        }

    # --------------------------------------------------
    # Autocorrelation dataframe
    # --------------------------------------------------

    df_autocorr = pd.DataFrame({
        "lag_samples": lags,
        "lag_time": lags_time,
        "autocorr_x": autocorr_x,
        "autocorr_y": autocorr_y,
        "autocorr_x_centered": autocorr_x_centered,
        "autocorr_y_centered": autocorr_y_centered
    })

    # --------------------------------------------------
    # Frequency-domain dataframe
    # --------------------------------------------------

    df_freq = pd.DataFrame({
        "frequency": fxx,

        "pn_x": pn_x,
        "pn_y": pn_y,
        "pn_xy": pn_xy,

        "fn_x": fn_x,
        "fn_y": fn_y,
        "fn_xy": fn_xy
    })


    if Nick_analysis:
        return {
            "time_domain": df_time,
            "autocorrelation": df_autocorr,
            "frequency_domain": df_freq,
            "variance_summary": df_variance,
            "variance_details": variance_details,
            "meta data": meta,
            "FSR": FSR,
            "FSR_peaks_freq": peaks_freq,
            "FSR_peaks_psd": peaks_psd,
            "Delay": delay,
            "Averaging number": averaging_number
        }
    else:
        return {
            "time_domain": df_time,
            "autocorrelation": df_autocorr,
            "frequency_domain": df_freq,
            "meta data": meta,
            "FSR": FSR,
            "FSR_peaks_freq": peaks_freq,
            "FSR_peaks_psd": peaks_psd,
            "Delay": delay,
            "Averaging number": averaging_number
        }

In [50]:
def plot_lissajous(
        label_name,
        time_domain_df,
        polarization="x",
        corrected=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(8,8))

    if polarization.lower() == "x":

        I = (
            time_domain_df["Ix_corr"]
            if corrected
            else time_domain_df["Ix_raw"]
        )

        Q = (
            time_domain_df["Qx_corr"]
            if corrected
            else time_domain_df["Qx_raw"]
        )

        pol_label = "X"

    else:

        I = (
            time_domain_df["Iy_corr"]
            if corrected
            else time_domain_df["Iy_raw"]
        )

        Q = (
            time_domain_df["Qy_corr"]
            if corrected
            else time_domain_df["Qy_raw"]
        )

        pol_label = "Y"

    ax.plot(
        I,
        Q,
        ".",
        ms=1,
        label=f"{pol_label}-pol {label_name}"
    )

    if plot_settings:
        ax.set_xlabel("I [V]")
        ax.set_ylabel("Q [V]")
        ax.set_title(
            f"Lissajous Diagram ({pol_label}-pol)"
        )
        ax.axis("equal")
        ax.grid(True)
        ax.legend(loc="upper right")

    return ax


def plot_phase_increment(
        label_name,
        time_domain_df,
        plot_x=True,
        plot_y=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    if plot_x:
        ax.plot(
            time_domain_df["time"],
            time_domain_df["phase_inc_x"],
            label=f"X-pol {label_name}"
        )

    if plot_y:
        ax.plot(
            time_domain_df["time"],
            time_domain_df["phase_inc_y"],
            label=f"Y-pol {label_name}"
        )

    if plot_settings:
        ax.set_xlabel("Time [s]")
        ax.set_ylabel("Phase increment [rad]")

        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right")

    return ax

def plot_phase_increment_zoom(
        label_name,
        time_domain_df,
        start_time=0.5,
        duration=100e-6,
        plot_x=True,
        plot_y=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))

    end_time = start_time + duration

    mask = (
        (time_domain_df["time"] >= start_time)
        &
        (time_domain_df["time"] <= end_time)
    )

    if plot_x:
        ax.plot(
            time_domain_df.loc[mask, "time"],
            time_domain_df.loc[mask, "phase_inc_x"],
            label=f"X-pol {label_name}"
        )

    if plot_y:
        ax.plot(
            time_domain_df.loc[mask, "time"],
            time_domain_df.loc[mask, "phase_inc_y"],
            label=f"Y-pol {label_name}"
        )

    if plot_settings:

        ax.set_xlabel("Time [s]")
        ax.set_ylabel("Phase increment [rad]")

        ax.set_title(
            f"Phase increment zoom ({duration*1e6:.0f} µs)"
        )

        ax.grid(True)
        ax.legend(loc="upper right")

    return ax


def plot_variance_statistics(
        label_name,
        variance_summary_df,
        plot_x=True,
        plot_y=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))

    if plot_x:
        ax.errorbar(
            variance_summary_df["Twin_ms"],
            variance_summary_df["mean_var_x"],
            variance_summary_df["std_var_x"],
            fmt='o-',
            label=f"X-pol {label_name}"
        )

    if plot_y:
        ax.errorbar(
            variance_summary_df["Twin_ms"],
            variance_summary_df["mean_var_y"],
            variance_summary_df["std_var_y"],
            fmt='o-',
            label=f"Y-pol {label_name}"
        )

    if plot_settings:

        ax.set_xscale("log")

        ax.set_xlabel(
            r"Observation window $T_{win}$ [ms]"
        )

        ax.set_ylabel(
            r"Variance of $\Delta\phi$ [rad$^2$]"
        )

        ax.grid(True)
        ax.legend(loc="upper right")

    return ax


def plot_variance_stability(
        label_name,
        variance_summary_df,
        plot_x=True,
        plot_y=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))

    if plot_x:
        ax.plot(
            variance_summary_df["Twin_ms"],
            variance_summary_df["std_var_x"],
            'o-',
            label=f"X-pol {label_name}"
        )

    if plot_y:
        ax.plot(
            variance_summary_df["Twin_ms"],
            variance_summary_df["std_var_y"],
            'o-',
            label=f"Y-pol {label_name}"
        )

    if plot_settings:

        ax.set_xscale("log")

        ax.set_xlabel(
            r"Observation window $T_{win}$ [ms]"
        )

        ax.set_ylabel(
            r"Std. dev. of local variances [rad$^2$]"
        )

        ax.grid(True)
        ax.legend(loc="upper right")

    return ax


def plot_variance_vs_start_time(
        label_name,
        variance_details,
        Twin_target=1e-3,
        plot_x=True,
        plot_y=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))

    Twin_list = variance_details["Twin_list"]

    idx = np.argmin(
        np.abs(Twin_list - Twin_target)
    )

    if plot_x:

        ax.plot(
            variance_details["start_times_x"][idx]*1e3,
            variance_details["local_variances_x"][idx],
            label=f"X-pol {label_name}"
        )

    if plot_y:

        ax.plot(
            variance_details["start_times_y"][idx]*1e3,
            variance_details["local_variances_y"][idx],
            label=f"Y-pol {label_name}"
        )

    if plot_settings:

        ax.set_xlabel(
            "Window start time [ms]"
        )

        ax.set_ylabel(
            r"Variance of $\Delta\phi$ [rad$^2$]"
        )

        ax.set_title(
            f"Twin = {Twin_list[idx]*1e3:.2f} ms"
        )

        ax.grid(True)
        ax.legend(loc="upper right")

    return ax

def plot_phase_increment_histograms(
        label_name,
        time_domain_df,
        Twin=2e-3,
        polarization="x",
        start_times=None,
        ax=None,
        plot_settings=True):

    if start_times is None:

        start_times = [
            0e-3,
            100e-3,
            200e-3,
            300e-3
        ]

    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))

    time = time_domain_df["time"]

    phase = (
        time_domain_df["phase_inc_x"]
        if polarization.lower() == "x"
        else time_domain_df["phase_inc_y"]
    )

    for t0 in start_times:

        mask = (
            (time >= t0)
            &
            (time < t0 + Twin)
        )

        ax.hist(
            phase[mask],
            bins=40,
            density=True,
            histtype="step",
            linewidth=2,
            label=f"{t0*1e3:.0f} ms"
        )

    if plot_settings:

        ax.set_xlabel(r"$\Delta\phi$ [rad]")

        ax.set_ylabel(
            "Probability density"
        )

        ax.set_title(
            f"{label_name} - "
            f"{polarization.upper()} polarization "
            f"(Twin = {Twin*1e3:.1f} ms)"
        )

        ax.grid(True)
        ax.legend(loc="upper right")

    return ax


def plot_autocorrelation(
        label_name,
        autocorrelation_df,
        plot_x=True,
        plot_y=True,
        centered=False,
        use_time_lag=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    x_axis = (
        autocorrelation_df["lag_time"]
        if use_time_lag
        else autocorrelation_df["lag_samples"]
    )

    x_label = (
        "Lag time [s]"
        if use_time_lag
        else "Lag samples"
    )

    x_col = "autocorr_x_centered" if centered else "autocorr_x"
    y_col = "autocorr_y_centered" if centered else "autocorr_y"

    if plot_x:
        ax.plot(
            x_axis,
            autocorrelation_df[x_col],
            label=f"X-pol {label_name}"
        )

    if plot_y:
        ax.plot(
            x_axis,
            autocorrelation_df[y_col],
            label=f"Y-pol {label_name}"
        )

    if plot_settings:
        ax.set_xlabel(x_label)
        ax.set_ylabel(r"Autocorrelation centered [rad$^2$]" if centered else "Autocorrelation [rad$^2$]")
        ax.grid(True)
        ax.legend(loc="upper right")

    return ax




def plot_pnpsd(
        label_name,
        frequency_domain_df,
        plot_x=True,
        plot_y=True,
        plot_xy=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    freq = frequency_domain_df["frequency"]


    if plot_xy:
        ax.loglog(
            freq,
            frequency_domain_df["pn_xy"],
            label=f"XY-pol {label_name}"
        )

        
    if plot_y:
        ax.loglog(
            freq,
            frequency_domain_df["pn_y"],
            label=f"Y-pol {label_name}"
        )

    if plot_x:
        ax.loglog(
            freq,
            frequency_domain_df["pn_x"],
            label=f"X-pol {label_name}"
        )

    if plot_settings:

        ax.xaxis.set_major_formatter(
            FuncFormatter(engineering_formatter)
        )

        ax.set_xlabel("Fourier frequency [Hz]")
        ax.set_ylabel(r"PN PSD [rad$^2$/Hz]")

        ax.grid(True, which="both", alpha=0.3)
        ax.legend(loc="upper right")

    return ax


def plot_fsr_detection(
        label_name,
        result,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10,6))

    freq_df = result["frequency_domain"]

    ax.loglog(
        freq_df["frequency"],
        freq_df["pn_x"],
        label=f"PN PSD {label_name}"
    )

    ax.scatter(
        result["FSR_peaks_freq"],
        result["FSR_peaks_psd"],
        color="red",
        label="Detected fringes"
    )

    if plot_settings:

        ax.set_xlabel(
            "Fourier frequency [Hz]"
        )

        ax.set_ylabel(
            r"PN PSD [rad$^2$/Hz]"
        )

        ax.set_title(
            f"FSR = {result['FSR'] / 1e3:.1f} kHz"
        )

        ax.grid(True)
        ax.legend()

    return ax



def plot_fnpsd(
        label_name,
        frequency_domain_df,
        plot_x=True,
        plot_y=True,
        plot_xy=True,
        ax=None,
        plot_settings=True):

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))



    commercial_files = {
    "Whisper": r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Whisper\HighFinesse\PSD_PPCL_Whisper_14.66dBm_no3.txt",
    "Dither": r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Dither\HighFinesse\PSD_PPCL_Dither_14.66dBm_no4.txt",
    "NKT": r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT\HighFinesse\14.66 dBm\PSD_NKT_14.66dBm_no7.txt",
    "Agilent": r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\Agilent 81940A\HighFinesse\PSD_Agilent_81940A_14.771dBm_no4.txt"
    }

    if "Whisper" in label_name:
        commercial_file = commercial_files["Whisper"]

    elif "Dither" in label_name:
        commercial_file = commercial_files["Dither"]

    elif "NKT" in label_name:
        commercial_file = commercial_files["NKT"]

    elif "Agilent" in label_name:
        commercial_file = commercial_files["Agilent"]

    else:
        commercial_file = None

    if commercial_file is not None:


        if commercial_file not in COMMERCIAL_CACHE:

            COMMERCIAL_CACHE[commercial_file] = load_commercial_fn(commercial_file)



        commercial_freq, commercial_fn = COMMERCIAL_CACHE[commercial_file]           

        ax.loglog(commercial_freq, commercial_fn, label=f"HighFinesse", color="black", alpha=0.5)



    freq = frequency_domain_df["frequency"]

    if plot_xy:
        ax.loglog(
            freq,
            frequency_domain_df["fn_xy"],
            label=f"XY-pol {label_name}"
        )


    if plot_y:
        ax.loglog(
            freq,
            frequency_domain_df["fn_y"],
            label=f"Y-pol {label_name}"
        )


    if plot_x:
        ax.loglog(
            freq,
            frequency_domain_df["fn_x"],
            label=f"X-pol {label_name}"
        )


    if plot_settings:

        ax.xaxis.set_major_formatter(
            FuncFormatter(engineering_formatter)
        )

        ax.set_xlabel("Fourier frequency [Hz]")
        ax.set_ylabel(r"FN PSD [Hz$^2$/Hz]")

        ax.grid(True, which="both", alpha=0.3)
        ax.legend(loc="upper right")


        mask = ( (freq> 0) & (freq <= 1e4))

        mask_commercial = ( (commercial_freq> 0) & (commercial_freq <= 1e4))

        if commercial_file is not None:                     
            vals_max = np.concatenate([
                frequency_domain_df.loc[mask, "fn_x"].values,
                frequency_domain_df.loc[mask, "fn_y"].values,
                frequency_domain_df.loc[mask, "fn_xy"].values,
                commercial_fn[mask_commercial]
            ])

            vals_min = np.concatenate([
                frequency_domain_df["fn_x"],
                frequency_domain_df["fn_y"],
                frequency_domain_df["fn_xy"],
                commercial_fn
                ])           
        else:
            vals_max = np.concatenate([
                frequency_domain_df.loc[mask, "fn_x"].values,
                frequency_domain_df.loc[mask, "fn_y"].values,
                frequency_domain_df.loc[mask, "fn_xy"].values
            ])

            vals_min = np.concatenate([
                frequency_domain_df["fn_x"],
                frequency_domain_df["fn_y"],
                frequency_domain_df["fn_xy"]
            ])                                                   

        vals_max = vals_max[
            np.isfinite(vals_max)
            &
            (vals_max > 0)
        ]

        vals_min = vals_min[
                    np.isfinite(vals_min)
                    &
                    (vals_min > 0)
                ]

        if len(vals_max) > 0:
            ymin = np.min(vals_min) / 10
            ymax = np.max(vals_max) * 10

            ax.set_ylim(ymin, ymax)
        ax.set_xlim(1e2, 1e7)

    return ax

In [51]:
def save_figure(fig, folder, basename, suffix):
    """
    Save figure as PDF and SVG.
    """

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        folder / f"{basename}_{suffix}.png",dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        folder / f"{basename}_{suffix}.svg",
        bbox_inches="tight"
    )

    # drawing = svg2rlg(folder / f"{basename}_{suffix}.svg")
    # renderPDF.drawToFile(drawing, str(folder / f"{basename}_{suffix}.pdf") )

In [52]:
def plot_analysis_overview(
        label_name,
        result, save_folder=None, Nick_analysis=False):

    basename = Path(result["meta data"]["MeasurementName"]).stem
    avg_no = result["Averaging number"]
    delay = result["Delay"]

    basename += f"_avg_no_{avg_no}"
    basename += f"_delay_{delay}"

    # ----------------------------------
    # Lissajous X polarization
    # ----------------------------------

    fig, ax = plt.subplots(figsize=(8, 8))

    plot_lissajous(
        label_name,
        result["time_domain"],
        polarization="x",
        corrected=True,
        ax=ax,
        plot_settings=True
    )

    if save_folder is not None:
    
        save_figure(fig,save_folder,basename,"lissajous_x")

        plt.close(ax.figure)

    # ----------------------------------
    # Lissajous Y polarization
    # ----------------------------------


    fig, ax = plt.subplots(figsize=(8, 8))

    plot_lissajous(
        label_name,
        result["time_domain"],
        polarization="y",
        corrected=True,
        ax=ax,
        plot_settings=True
    )

    if save_folder is not None:
    
        save_figure(fig,save_folder,basename,"lissajous_y")

        plt.close(ax.figure)

    # ----------------------------------
    # Phase increment
    # ----------------------------------

    fig, ax = plt.subplots(figsize=(10, 6))

    plot_phase_increment(
        label_name,
        result["time_domain"],
        ax=ax,
        plot_settings=True
    )
    if save_folder is not None:
        
        save_figure(fig,save_folder,basename,"phase_increment")

        plt.close(ax.figure)

    # ----------------------------------
    # Phase increment zoom
    # ----------------------------------

    fig, ax = plt.subplots(figsize=(10, 6))

    plot_phase_increment_zoom(
        label_name,
        result["time_domain"],
        ax=ax,
        start_time=0.5,
        duration=100e-6,
        plot_settings=True
    )
    if save_folder is not None:
        
        save_figure(fig,save_folder,basename,"phase_increment_zoom")

        plt.close(ax.figure)

    if Nick_analysis:
        # ----------------------------------
        # Histograms
        # Twin = 0.2 ms
        # ----------------------------------

        for pol in ["x", "y"]:

            fig, ax = plt.subplots(figsize=(10, 6))

            plot_phase_increment_histograms(
                label_name,
                result["time_domain"],
                Twin=0.2e-3,
                polarization=pol,
                ax=ax,
                plot_settings=True
            )

            if save_folder is not None:

                save_figure(
                    fig,
                    save_folder,
                    basename,
                    f"histogram_0p2ms_{pol}"
                )

                plt.close(ax.figure)


        # ----------------------------------
        # Histograms
        # Twin = 2 ms
        # ----------------------------------

        for pol in ["x", "y"]:

            fig, ax = plt.subplots(figsize=(10, 6))

            plot_phase_increment_histograms(
                label_name,
                result["time_domain"],
                Twin=2e-3,
                polarization=pol,
                ax=ax,
                plot_settings=True
            )

            if save_folder is not None:

                save_figure(
                    fig,
                    save_folder,
                    basename,
                    f"histogram_2ms_{pol}"
                )

                plt.close(ax.figure)


        # ----------------------------------
        # Histograms
        # Twin = 20 ms
        # ----------------------------------

        for pol in ["x", "y"]:

            fig, ax = plt.subplots(figsize=(10, 6))

            plot_phase_increment_histograms(
                label_name,
                result["time_domain"],
                Twin=20e-3,
                polarization=pol,
                ax=ax,
                plot_settings=True
            )

            if save_folder is not None:

                save_figure(
                    fig,
                    save_folder,
                    basename,
                    f"histogram_20ms_{pol}"
                )

                plt.close(ax.figure)

            # ----------------------------------
        # Variance statistics (Fig. 6 type)
        # ----------------------------------

        fig, ax = plt.subplots(figsize=(10, 6))

        plot_variance_statistics(
            label_name,
            result["variance_summary"],
            plot_x=True,
            plot_y=True,
            ax=ax,
            plot_settings=True
        )

        if save_folder is not None:

            save_figure(
                fig,
                save_folder,
                basename,
                "variance_statistics"
            )

            plt.close(ax.figure)

        # ----------------------------------
        # Variance stability (Fig. 7 type)
        # ----------------------------------

        fig, ax = plt.subplots(figsize=(10, 6))

        plot_variance_stability(
            label_name,
            result["variance_summary"],
            plot_x=True,
            plot_y=True,
            ax=ax,
            plot_settings=True
        )



        if save_folder is not None:

            save_figure(
                fig,
                save_folder,
                basename,
                "variance_stability"
            )

            plt.close(ax.figure)

        # ----------------------------------
        # Variance vs start time
        # Twin = 0.1 ms
        # ----------------------------------

        fig, ax = plt.subplots(figsize=(10, 6))

        plot_variance_vs_start_time(
            label_name,
            result["variance_details"],
            Twin_target=0.1e-3,
            plot_x=True,
            plot_y=True,
            ax=ax,
            plot_settings=True
        )



        if save_folder is not None:

            save_figure(
                fig,
                save_folder,
                basename,
                "variance_vs_start_time_0p1ms"
            )

            plt.close(ax.figure)

        # ----------------------------------
        # Variance vs start time
        # Twin = 1 ms
        # ----------------------------------

        fig, ax = plt.subplots(figsize=(10, 6))

        plot_variance_vs_start_time(
            label_name,
            result["variance_details"],
            Twin_target=1e-3,
            plot_x=True,
            plot_y=True,
            ax=ax,
            plot_settings=True
        )

        if save_folder is not None:

            save_figure(
                fig,
                save_folder,
                basename,
                "variance_vs_start_time_1ms"
            )

            plt.close(ax.figure)

        # ----------------------------------
        # Variance vs start time
        # Twin = 10 ms
        # ----------------------------------

        fig, ax = plt.subplots(figsize=(10, 6))

        plot_variance_vs_start_time(
            label_name,
            result["variance_details"],
            Twin_target=10e-3,
            plot_x=True,
            plot_y=True,
            ax=ax,
            plot_settings=True
        )

        if save_folder is not None:

            save_figure(
                fig,
                save_folder,
                basename,
                "variance_vs_start_time_10ms"
            )

            plt.close(ax.figure)



    # ----------------------------------
    # Raw autocorrelation
    # ----------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))

    plot_autocorrelation(
        label_name,
        result["autocorrelation"],
        centered=False,
        ax=ax,
        plot_settings=True
    )

    if save_folder is not None:
        
        save_figure(fig,save_folder,basename,"autocorrelation")

        plt.close(ax.figure)

    # ----------------------------------
    # Centered autocorrelation
    # ----------------------------------

    fig, ax = plt.subplots(figsize=(10, 6))

    plot_autocorrelation(
        label_name,
        result["autocorrelation"],
        centered=True,
        ax=ax,
        plot_settings=True
    )

    if save_folder is not None:
        
        save_figure(fig,save_folder,basename,"autocorrelation_centered")

        plt.close(ax.figure)


    # ----------------------------------
    # PN PSD
    # ----------------------------------

    fig, ax = plt.subplots(figsize=(10, 6))

    plot_pnpsd(
        label_name,
        result["frequency_domain"],
        ax=ax,
        plot_settings=True
    )

    if save_folder is not None:
        save_figure(fig,save_folder,basename,"pn_psd")

        plt.close(ax.figure)

    # ----------------------------------
    # FSR detection
    # ----------------------------------

    fig, ax = plt.subplots(figsize=(10,6))

    plot_fsr_detection(
        label_name,
        result,
        ax=ax,
        plot_settings=True
    )

    if save_folder is not None:

        save_figure(
            fig,
            save_folder,
            basename,
            "fsr_detection"
        )

        plt.close(ax.figure)

    # ----------------------------------
    # FN PSD
    # ----------------------------------

    fig, ax = plt.subplots(figsize=(10, 6))

    plot_fnpsd(
        label_name,
        result["frequency_domain"],
        ax=ax,
        plot_settings=True
    )
    if save_folder is not None:
        save_figure(fig,save_folder,basename,"fn_psd")

        plt.close(ax.figure)

## For quick examination of data, simply use the pnpsd function

## If the QAM is acting up, use pnpsd_noQAM instead

In [ ]:
averages = 20

folder_dither = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Dither 700m delay"
folder_whisper = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Whisper 700m delay"


data_PPCL = {"Whisper": {}, "Dither": {}}


for mode in ["Whisper","Dither"]:

    folder = folder_whisper if mode == "Whisper" else folder_dither

    for i in range(1, 6):
        name = f"PPCL550_{mode}_14.66dBm_7.51+12.14dBm_no{i}.Wfm"
        
        data = analyze_data(name, folder, averages,delay = 700, offset_handling=False)

        data_PPCL[mode]["No. "+str(i)] = data 


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Whisper 700m delay/PPCL550_Whisper_14.66dBm_7.51+12.14dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Whisper 700m delay/PPCL550_Whisper_14.66dBm_7.51+12.14dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0


In [ ]:
averages = 20


for mode in ["Whisper", "Dither"]:    
    for i in range(1, 6):

        key = f"No. {i}"

        savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026_HHI_figures\PPCL {mode}\No_{i}\Averages_{averages}"

        plot_analysis_overview(label_name=f"PPCL {mode} No. {i}", result=data_PPCL[mode][key], save_folder=savefolder)


        del data_PPCL[mode][key]

        gc.collect()

        print(f"Finished {mode} No. {i}")




C:\Users\au617810\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\xy\components.py:5282: RuntimeWarning: scatter has 20,000,000 points above the soft ceiling (2,000,000); using a density surface for the initial render.
  fig.scatter(


Finished Whisper No. 1
Finished Whisper No. 2
Finished Whisper No. 3
Finished Whisper No. 4
Finished Whisper No. 5
Finished Dither No. 1
Finished Dither No. 2
Finished Dither No. 3
Finished Dither No. 4
Finished Dither No. 5


In [ ]:
folder_NKT = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay"

delay = 700


data_NKT = {}


folder = folder_NKT

for i in range(1,6):
    name = f"NKT_14.66dBm_7.57+12.08dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages,delay = delay, offset_handling=False)

    data_NKT["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay/NKT_14.66dBm_7.57+12.08dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay/NKT_14.66dBm_7.57+12.08dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 33 fringe dips.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay/NKT_14.66dBm_7.57+12.08dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay/NKT_14.66dBm_7.57+12.08dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 32 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay/NKT_14.66dBm_7.57+12.08dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay/NKT_14.66dBm_7.57+12.08dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 35 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent rec

In [ ]:
for i in range(1,6):

    key = f"No. {i}"

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\NKT\NKT_delay_{delay}m\No_{i}\Averages_{averages}"

    plot_analysis_overview(label_name=f"NKT No. {i}", result=data_NKT[key], save_folder=savefolder)


    del data_NKT[key]

    gc.collect()

    print(f"Finished NKT No. {i}")





Finished NKT No. 1
Finished NKT No. 2
Finished NKT No. 3
Finished NKT No. 4
Finished NKT No. 5


In [ ]:
folder_agilent = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 700m"

delay = 700


data_agilent = {}


folder = folder_agilent

for i in range(1,6):
    name = f"Agilent_14.771dBm_6.45+10.99dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages,delay = delay, offset_handling=False)

    data_agilent["No. "+str(i)] = data 






C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 700m/Agilent_14.771dBm_6.45+10.99dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 700m/Agilent_14.771dBm_6.45+10.99dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 30 fringe dips.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 700m/Agilent_14.771dBm_6.45+10.99dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 700m/Agilent_14.771dBm_6.45+10.99dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 31 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 700m/Agilent_14.771dBm_6.45+10.99dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 700m/Agilent_14.771dBm_6.45+10.99dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 31 fringe dips.
C:\Users\au617810\OneDrive - Aarhus 

In [ ]:
for i in range(1,6):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\Agilent 81940A\Agilent_delay_{delay}m\No_{i}\Averages_{averages}"


    key = f"No. {i}"

    plot_analysis_overview(label_name=f"Agilent No. {i}", result=data_agilent[key], save_folder=savefolder)

    del data_agilent[key]

    gc.collect()

    print(f"Finished Agilent No. {i}")



Finished Agilent No. 1
Finished Agilent No. 2
Finished Agilent No. 3
Finished Agilent No. 4
Finished Agilent No. 5


In [ ]:
folder_NKT_3km = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 3km"


# folder_NKT = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 700m delay"

delay = 3000


data_NKT_3km = {}


folder = folder_NKT_3km

for i in range(1, 6):
    name = f"NKT_14.66dBm_5.49+12.04dBm_no{i}.Wfm"

   
    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_NKT_3km["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 3km/NKT_14.66dBm_5.49+12.04dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 3km/NKT_14.66dBm_5.49+12.04dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 140 fringe dips.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 3km/NKT_14.66dBm_5.49+12.04dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 3km/NKT_14.66dBm_5.49+12.04dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 140 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 3km/NKT_14.66dBm_5.49+12.04dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 3km/NKT_14.66dBm_5.49+12.04dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 142 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Co

In [ ]:
for i in range(1, 6):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\NKT\NKT_delay_{delay}m\No_{i}\Averages_{averages}"
    key = f"No. {i}"
    
    plot_analysis_overview(label_name=f"NKT No. {i}", result=data_NKT_3km[key], save_folder=savefolder)

    del data_NKT_3km[key]

    gc.collect()

    print(f"Finished NKT No. {i}")



Finished NKT No. 1
Finished NKT No. 2
Finished NKT No. 3
Finished NKT No. 4
Finished NKT No. 5


In [ ]:
folder_NKT_30m = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 30m"


delay = 30


data_NKT_30m = {}


folder = folder_NKT_30m
    
for i in range(1,6):
    name = f"NKT_14.66dBm_7.54+12.04dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_NKT_30m["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 30m/NKT_14.66dBm_7.54+12.04dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 30m/NKT_14.66dBm_7.54+12.04dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 30m/NKT_14.66dBm_7.54+12.04dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 30m/NKT_14.66dBm_7.54+12.04dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 30m/NKT_14.66dBm_7.54+12.04dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 14.66dBm 30m/NKT_14.66dBm_7.54+12.04dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detect

In [ ]:
for i in range(1,6):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\NKT\NKT_delay_{delay}m\No_{i}\Averages_{averages}"

    key = f"No. {i}"

    plot_analysis_overview(label_name=f"NKT No. {i}", result=data_NKT_30m[key], save_folder=savefolder)

    del data_NKT_30m[key]

    gc.collect()

    print(f"Finished NKT No. {i}")



Finished NKT No. 1
Finished NKT No. 2
Finished NKT No. 3
Finished NKT No. 4
Finished NKT No. 5


In [ ]:
folder_NKT_10km = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 10km"


delay = 10000


data_NKT_10km = {}


folder = folder_NKT_10km

for i in range(1, 5):
    name = f"NKT_14.66dBm_7.74+10.66dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_NKT_10km["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 10km/NKT_14.66dBm_7.74+10.66dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 10km/NKT_14.66dBm_7.74+10.66dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 476 fringe dips.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 10km/NKT_14.66dBm_7.74+10.66dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 10km/NKT_14.66dBm_7.74+10.66dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 492 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 10km/NKT_14.66dBm_7.74+10.66dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\NKT 10km/NKT_14.66dBm_7.74+10.66dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 483 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI D

In [ ]:
for i in range(1, 5):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\NKT\NKT_delay_{delay}m\No_{i}\Averages_{averages}"

    key = f"No. {i}"
    
    plot_analysis_overview(label_name=f"NKT No. {i}", result=data_NKT_10km[key], save_folder=savefolder)

    del data_NKT_10km[key]

    gc.collect()

    print(f"Finished NKT No. {i}")




Finished NKT No. 1
Finished NKT No. 2
Finished NKT No. 3
Finished NKT No. 4


In [ ]:
folder_PPCL_Whisper_30m = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 30m"


delay = 30


data_PPCL_Whisper_30m = {}


folder = folder_PPCL_Whisper_30m

for i in range(1,4):
    name = f"PPCL550_Whisper_14.66dBm_7.65+12.26dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_PPCL_Whisper_30m["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 30m/PPCL550_Whisper_14.66dBm_7.65+12.26dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 30m/PPCL550_Whisper_14.66dBm_7.65+12.26dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 30m/PPCL550_Whisper_14.66dBm_7.65+12.26dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 30m/PPCL550_Whisper_14.66dBm_7.65+12.26dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 30m/PPCL550_Whisper_14.66dBm_7.65+12.26dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 30m/PPCL550_Whisper_14.66dBm_7.65+12.26dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  200000

In [ ]:
for i in range(1,4):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\PPCL Whisper\PPCL_Whisper_delay_{delay}m\No_{i}\Averages_{averages}"

    key = f"No. {i}"

    plot_analysis_overview(label_name=f"PPCL Whisper No. {i}", result=data_PPCL_Whisper_30m[key], save_folder=savefolder)

    del data_PPCL_Whisper_30m[key]

    gc.collect()

    print(f"Finished PPCL Whisper No. {i}")



Finished PPCL Whisper No. 1
Finished PPCL Whisper No. 2
Finished PPCL Whisper No. 3


In [ ]:
folder_PPCL_Dither_30m = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 30m"


delay = 30


data_PPCL_Dither_30m = {}

folder = folder_PPCL_Dither_30m

for i in range(1,4):
    name = f"PPCL550_Dither_14.66dBm_7.51+12.14dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_PPCL_Dither_30m["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 30m/PPCL550_Dither_14.66dBm_7.51+12.14dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 30m/PPCL550_Dither_14.66dBm_7.51+12.14dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 30m/PPCL550_Dither_14.66dBm_7.51+12.14dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 30m/PPCL550_Dither_14.66dBm_7.51+12.14dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 30m/PPCL550_Dither_14.66dBm_7.51+12.14dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 30m/PPCL550_Dither_14.66dBm_7.51+12.14dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Det

In [ ]:
for i in range(1, 4):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\PPCL Dither\PPCL_Dither_delay_{delay}m\No_{i}\Averages_{averages}"

    key = f"No. {i}"

    plot_analysis_overview(label_name=f"PPCL Dither No. {i}", result=data_PPCL_Dither_30m[key], save_folder=savefolder)


    del data_PPCL_Dither_30m[key]

    gc.collect()

    print(f"Finished PPCL Dither No. {i}")


Finished PPCL Dither No. 1
Finished PPCL Dither No. 2
Finished PPCL Dither No. 3


In [ ]:
folder_PPCL_Dither_3km = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 3km"


delay = 3000


data_PPCL_Dither_3km = {}


folder = folder_PPCL_Dither_3km

for i in range(1, 4):
    name = f"PPCL550_Dither_14.66dBm_5.99+12.28dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_PPCL_Dither_3km["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 3km/PPCL550_Dither_14.66dBm_5.99+12.28dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 3km/PPCL550_Dither_14.66dBm_5.99+12.28dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 3km/PPCL550_Dither_14.66dBm_5.99+12.28dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 3km/PPCL550_Dither_14.66dBm_5.99+12.28dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 3km/PPCL550_Dither_14.66dBm_5.99+12.28dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Dither 3km/PPCL550_Dither_14.66dBm_5.99+12.28dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.


In [ ]:
for i in range(1, 4):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\PPCL Dither\PPCL_Dither_delay_{delay}m\No_{i}\Averages_{averages}"

    key = f"No. {i}"
    plot_analysis_overview(label_name=f"PPCL Dither No. {i}", result=data_PPCL_Dither_3km[key], save_folder=savefolder)

    del data_PPCL_Dither_3km[key]

    gc.collect()

    print(f"Finished PPCL Dither No. {i}")



Finished PPCL Dither No. 1
Finished PPCL Dither No. 2
Finished PPCL Dither No. 3


In [ ]:
folder_PPCL_Whisper_3km = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 3km"


delay = 3000


data_PPCL_Whisper_3km = {}


folder = folder_PPCL_Whisper_3km

for i in range(1, 4):
    name = f"PPCL550_Whisper_14.66dBm_5.99+12.28dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_PPCL_Whisper_3km["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 3km/PPCL550_Whisper_14.66dBm_5.99+12.28dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 3km/PPCL550_Whisper_14.66dBm_5.99+12.28dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 3km/PPCL550_Whisper_14.66dBm_5.99+12.28dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 3km/PPCL550_Whisper_14.66dBm_5.99+12.28dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 3km/PPCL550_Whisper_14.66dBm_5.99+12.28dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\PPCL Whisper 3km/PPCL550_Whisper_14.66dBm_5.99+12.28dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.


In [ ]:
for i in range(1, 4):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\PPCL Whisper\PPCL_Whisper_delay_{delay}m\No_{i}\Averages_{averages}"


    key = f"No. {i}"

    plot_analysis_overview(label_name=f"PPCL Whisper No. {i}", result=data_PPCL_Whisper_3km[key], save_folder=savefolder)

    del data_PPCL_Whisper_3km[key]

    gc.collect()

    print(f"Finished PPCL Whisper No. {i}")



Finished PPCL Whisper No. 1
Finished PPCL Whisper No. 2
Finished PPCL Whisper No. 3


In [ ]:
folder_Agilent_3km = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 3km"


delay = 3000


data_Agilent_3km = {}


folder = folder_Agilent_3km

for i in range(1, 4):
    name = f"Agilent_14.771dBm_4.75+10.86dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_Agilent_3km["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 3km/Agilent_14.771dBm_4.75+10.86dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 3km/Agilent_14.771dBm_4.75+10.86dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 3km/Agilent_14.771dBm_4.75+10.86dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 3km/Agilent_14.771dBm_4.75+10.86dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 3km/Agilent_14.771dBm_4.75+10.86dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 3km/Agilent_14.771dBm_4.75+10.86dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 138 fringe dips.


In [ ]:
for i in range(1, 4):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\Agilent 81940A\Agilent_81940A_delay_{delay}m\No_{i}\Averages_{averages}"

    key = f"No. {i}"
    
    plot_analysis_overview(label_name=f"Agilent 81940A No. {i}", result=data_Agilent_3km[key], save_folder=savefolder)


    del data_Agilent_3km[key]

    gc.collect()

    print(f"Finished Agilent 81940A No. {i}")


Finished Agilent 81940A No. 1
Finished Agilent 81940A No. 2
Finished Agilent 81940A No. 3


In [ ]:
folder_Agilent_30m = r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 30m"


delay = 30


data_Agilent_30m = {}


folder = folder_Agilent_30m

for i in range(1, 4):
    name = f"Agilent_14.771dBm_6.47+10.87dBm_no{i}.Wfm"

    
    data = analyze_data(name, folder, averages, delay = delay, offset_handling=False)

    data_Agilent_30m["No. "+str(i)] = data 



C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 30m/Agilent_14.771dBm_6.47+10.87dBm_no1.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 30m/Agilent_14.771dBm_6.47+10.87dBm_no1.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.


C:\Users\au617810\AppData\Local\Temp\ipykernel_26764\415951329.py:255: RuntimeWarning: invalid value encountered in scalar divide
  fsdxy.append(fxy[i] ** 2 / (4 * np.sin(np.pi * fxy[i] * 1/FSR) ** 2) * psdxy[i])


C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 30m/Agilent_14.771dBm_6.47+10.87dBm_no2.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 30m/Agilent_14.771dBm_6.47+10.87dBm_no2.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 fringe dips.
Only one dip detected. Using dip frequency as FSR.
C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 30m/Agilent_14.771dBm_6.47+10.87dBm_no3.Wfm.csv
now proccessing file at path:  C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI Data\HHI Coherent Receiver\Agilent 81940A 30m/Agilent_14.771dBm_6.47+10.87dBm_no3.Wfm.csv
Size:  20000000
Frequency resolution:  20000000.0
Detected 1 frin

In [ ]:
for i in range(1, 4):

    savefolder = fr"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31_07_2026 HHI_figures\Agilent 81940A\Agilent_81940A_delay_{delay}m\No_{i}\Averages_{averages}"

    key = f"No. {i}"

    plot_analysis_overview(label_name=f"Agilent 81940A No. {i}", result=data_Agilent_30m[key], save_folder=savefolder)

    del data_Agilent_30m[key]

    gc.collect()

    print(f"Finished Agilent 81940A No. {i}")



Finished Agilent 81940A No. 1
Finished Agilent 81940A No. 2
Finished Agilent 81940A No. 3
